# Notebook 03: CTEs and Recursive Queries

**Phase 2 — CTEs & Recursive Queries (PostgreSQL)**

A Common Table Expression (CTE) is a named intermediate result defined with
`WITH name AS (...)` and referenced in the query that follows.  It does two
things a subquery cannot:

1. **Named reuse** — reference the same result multiple times in the same query
   without re-computing or re-stating it.
2. **Composability** — chain multiple named steps, each building on the last,
   so complex transformations read top-to-bottom rather than inside-out.

| Section | Concept |
|:---|:---|
| 1 | Chained CTEs — decomposing a 5-join aggregation |
| 2 | `MATERIALIZED` vs default inlining (PostgreSQL 12+) |
| 3 | Recursive CTEs — traversing a hierarchy |

---

## Prerequisites

Notebook 01 must have been run first (PostgreSQL seeded, Parquet files present).


In [ ]:
import pathlib
import sys

import psycopg2
import pandas as pd

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import settings

pg = psycopg2.connect(settings.dsn)
print(f"PostgreSQL connected: {settings.POSTGRES_HOST}:{settings.POSTGRES_PORT}/{settings.POSTGRES_DB}")


def run_query(sql: str) -> pd.DataFrame:
    with pg.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


SQL_DIR = ROOT / "sql" / "ctes"


def load_section(filename: str, section: str) -> str:
    text = (SQL_DIR / filename).read_text(encoding="utf-8")
    blocks: dict[str, str] = {}
    current: str | None = None
    acc: list[str] = []
    for line in text.splitlines():
        if line.startswith("-- §"):
            if current is not None:
                blocks[current] = "\n".join(acc).strip()
            current = line[4:].strip()
            acc = []
        else:
            acc.append(line)
    if current is not None:
        blocks[current] = "\n".join(acc).strip()
    if section not in blocks:
        raise KeyError(f"Section '{section}' not found in {filename}. Available: {list(blocks)}")
    return blocks[section]


---

## 1 · Chained CTEs

**Business question:** Which are the top 10 nations by net revenue from
high-priority orders shipped in 1995, and what share of their customers placed
at least one such order?

This requires filtering `lineitem`, joining to `orders`, `customer`, `nation`,
and `region`, aggregating at two levels, and computing a percentage — five
logical steps in one query.

### The subquery version (inside-out, hard to read)

The same query written as a nested subquery must be read from the innermost
bracket outward.  Modifications require tracking which closing parenthesis
belongs to which subquery.


In [ ]:
subquery_sql = """
SELECT
    region,
    nation,
    ROUND(total_net_revenue::NUMERIC, 2)  AS net_revenue_1995,
    contributing_customers,
    total_orders,
    ROUND(100.0 * contributing_customers / total_orders, 2)  AS pct_customers
FROM (
    SELECT
        cn.region,
        cn.nation,
        SUM(orv.order_net_revenue)        AS total_net_revenue,
        COUNT(DISTINCT orv.o_custkey)     AS contributing_customers,
        nt.total_orders
    FROM (
        SELECT o.o_custkey, SUM(l.l_extendedprice * (1 - l.l_discount)) AS order_net_revenue
        FROM lineitem l
        JOIN orders o ON l.l_orderkey = o.o_orderkey
        WHERE o.o_orderpriority IN ('1-URGENT', '2-HIGH')
          AND l.l_shipdate BETWEEN DATE '1995-01-01' AND DATE '1995-12-31'
        GROUP BY o.o_custkey
    ) orv
    JOIN (
        SELECT c.c_custkey, n.n_name AS nation, r.r_name AS region
        FROM customer c
        JOIN nation n ON c.c_nationkey = n.n_nationkey
        JOIN region r ON n.n_regionkey = r.r_regionkey
    ) cn ON orv.o_custkey = cn.c_custkey
    JOIN (
        SELECT n.n_name AS nation, COUNT(*) AS total_orders
        FROM orders o
        JOIN customer c ON o.o_custkey = c.c_custkey
        JOIN nation n ON c.c_nationkey = n.n_nationkey
        GROUP BY n.n_name
    ) nt ON cn.nation = nt.nation
    GROUP BY cn.region, cn.nation, nt.total_orders
) final
ORDER BY net_revenue_1995 DESC
LIMIT 10
"""

run_query(subquery_sql)


### The CTE version (top-to-bottom, each step named)

The same logic split into five named steps.  Each CTE is a complete,
readable unit — `customer_nation` only joins geography, `order_revenue` only
aggregates revenue.  The final `SELECT` reads like a summary of what each
step produced.

Note `nation_totals AS MATERIALIZED` — this CTE performs a full-table
`COUNT(*)` across all orders.  Marking it `MATERIALIZED` guarantees it
executes exactly once, storing the result before the planner begins work on
the filtered CTEs above it.  Without `MATERIALIZED`, PostgreSQL 12+ *may*
inline it and re-evaluate it inside the join — whether that is better or
worse depends on the planner's cost estimates.


In [ ]:
print(load_section("01_chained_ctes.sql", "chained_ctes"))


In [ ]:
df_chained = run_query(load_section("01_chained_ctes.sql", "chained_ctes"))
df_chained


---

## 2 · MATERIALIZED vs Default (PostgreSQL 12+)

| Behaviour | When | Effect |
|:---|:---|:---|
| Default (NOT MATERIALIZED) | PostgreSQL 12+ | Planner inlines CTE if referenced once and side-effect-free; can push predicates into it |
| `AS MATERIALIZED` | Any version | CTE always executes once; result stored in memory; outer predicates cannot filter inside it |
| `AS NOT MATERIALIZED` | Explicit opt-in | Force inlining even when planner would otherwise materialise (e.g. CTEs referenced multiple times) |

**When `MATERIALIZED` helps:** the CTE is expensive (large aggregation, heavy
join) and referenced more than once — materialise once, scan twice cheaply.

**When `MATERIALIZED` hurts:** the CTE has low selectivity and the outer query
adds a highly selective filter — inlining lets the planner push that filter
inside and use an index.

The query below demonstrates a case where *forcing* materialisation is
measurably worse: the date filter on `lineitem` can use an index when the CTE
is inlined, but not when materialised.


In [ ]:
# EXPLAIN (no ANALYZE — avoids executing 6M+ row scans just for the demo)
# Compare plan shapes: inlined CTE can use Index Scan; materialised cannot.

explain_inlined = """
EXPLAIN
WITH recent_lines AS (
    SELECT l_orderkey, l_extendedprice
    FROM   lineitem
    WHERE  l_shipdate = DATE '1998-09-01'
)
SELECT SUM(l_extendedprice) FROM recent_lines
"""

explain_materialised = """
EXPLAIN
WITH recent_lines AS MATERIALIZED (
    SELECT l_orderkey, l_extendedprice
    FROM   lineitem
    WHERE  l_shipdate = DATE '1998-09-01'
)
SELECT SUM(l_extendedprice) FROM recent_lines
"""

with pg.cursor() as cur:
    cur.execute(explain_inlined)
    print("--- Inlined (default) ---")
    for row in cur.fetchall():
        print(row[0])

    print()

    cur.execute(explain_materialised)
    print("--- MATERIALIZED ---")
    for row in cur.fetchall():
        print(row[0])


---

## 3 · Recursive CTEs

A recursive CTE adds `WITH RECURSIVE` and contains two parts joined by
`UNION ALL`:

```sql
WITH RECURSIVE cte AS (
    -- ANCHOR: seeds the recursion (non-recursive, executes once)
    SELECT ...

    UNION ALL

    -- RECURSIVE TERM: references cte itself; executes until no new rows
    SELECT ... FROM source JOIN cte ON ...
)
SELECT * FROM cte;
```

PostgreSQL executes the anchor once, then repeatedly executes the recursive
term — each pass using the rows produced by the *previous* pass — until the
join produces no new rows.

**Dataset:** `dim_employee` — a 4-level org hierarchy (see
`sql/schema/seed_employees.sql`).  10 rows, but the query patterns apply
identically to millions of nodes.


In [ ]:
# Step 1: basic traversal — walk the hierarchy from root to all leaves
print(load_section("02_recursive_cte.sql", "basic_recursion"))


In [ ]:
df_basic = run_query(load_section("02_recursive_cte.sql", "basic_recursion"))
df_basic


### Adding a depth counter

The anchor initialises `depth = 0` for the root.  Each recursive pass
increments `depth + 1`.  The result makes the level of each node explicit
and enables indented display.


In [ ]:
print(load_section("02_recursive_cte.sql", "depth_column"))


In [ ]:
df_depth = run_query(load_section("02_recursive_cte.sql", "depth_column"))
df_depth


### Accumulating a path string

At each recursive step the current employee's name is appended to the path
from the parent.  This gives every node its full lineage as a readable string:
`Alice Chen / Bob Patel / Dan Okafor / Grace Kim`.

A parallel `path_array` (integer array of visited IDs) is built alongside the
string — used for cycle detection in the next step.


In [ ]:
print(load_section("02_recursive_cte.sql", "path_accumulation"))


In [ ]:
df_path = run_query(load_section("02_recursive_cte.sql", "path_accumulation"))
df_path


### Cycle detection guard

In a clean database, FK constraints prevent cycles.  In real-world data
(scraped category trees, imported graph edges) cycles are common — and without
a guard, a cyclic graph causes **infinite recursion**.

The guard: add `WHERE e.id <> ALL(ot.path_array)` to the recursive term.
This skips any edge that would revisit an already-visited node.

`<> ALL(array)` is PostgreSQL's idiom for "not in this array" — equivalent to
`NOT (e.id = ANY(ot.path_array))`.

> **PostgreSQL 14+ alternative:** the SQL standard `CYCLE id SET is_cycle USING path`
> clause does this automatically, but the array guard is more portable and
> makes the mechanism explicit.


In [ ]:
print(load_section("02_recursive_cte.sql", "cycle_detection"))


In [ ]:
df_cycle = run_query(load_section("02_recursive_cte.sql", "cycle_detection"))
df_cycle


### Real-world patterns that use the same structure

The four queries above — basic traversal, depth, path, cycle guard — cover
the full recursive CTE toolkit.  The only thing that changes between use cases
is the table and the join predicate:

| Use case | Table shape | Join |
|:---|:---|:---|
| Org chart | `employee(id, manager_id)` | `e.manager_id = ot.id` |
| Category tree | `category(id, parent_id)` | `c.parent_id = ot.id` |
| Bill-of-materials | `component(part_id, sub_part_id, quantity)` | `c.part_id = ot.sub_part_id` |
| Network traversal | `edge(from_node, to_node)` | `e.from_node = ot.to_node` |


---

## Summary

| Concept | Key point |
|:---|:---|
| Named reuse | A CTE can be referenced multiple times; a subquery cannot |
| Readability | Chained CTEs express transformation steps top-to-bottom |
| `MATERIALIZED` | Forces one execution + stored result; use when CTE is expensive and referenced multiple times |
| Default (NOT MATERIALIZED) | Planner can inline and push predicates; better when outer filter is selective |
| Recursive anchor | Seeds the recursion; executes once |
| Recursive term | References the CTE itself; executes until no new rows |
| Depth counter | Increment on each recursive pass |
| Path accumulation | Append name/id arrays on each pass |
| Cycle guard | `WHERE id <> ALL(path_array)` prevents infinite recursion |

**Next:** [Notebook 04 — EXPLAIN Plans & Indexing](04_explain_plans.ipynb)
